# 03. Анализ графа

> **Краткое резюме для руководителя**: Этот ноутбук выполняет глубокий анализ отфильтрованного графа. Алгоритм Leiden находит группы тесно связанных компаний (кластеры/группы). Метрики центральности выявляют ключевых игроков и «привратников». Edge Score — комплексная оценка важности каждой транзакционной связи. Автоматическая классификация определяет роли узлов: head-компании, shell-компании, кондуиты. Обнаружение циклических платежей выявляет потенциальные схемы.

**User Story 3**: Кластеризация, метрики центральности, обнаружение shell-компаний и циклических платежей.

**Процесс**:
1. Загружаем отфильтрованный граф из pickle
2. Leiden кластеризация (CPM) с подбором gamma
3. Вычисление метрик центральности (PageRank, betweenness, clustering)
4. Вычисление edge_score для каждой транзакционной связи
5. Классификация ролей узлов (parent, subsidiary, conduit, shell, regular)
6. Обнаружение shell-компаний
7. Обнаружение циклических платежей
8. Сводка по кластерам (включая внешних контрагентов)

**Предпосылки**: Запустите `02_graph_construction.ipynb`.

---

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.graph_builder import compute_edge_score
from src.analysis import (
    run_leiden_clustering,
    compute_centrality,
    classify_node_roles,
    detect_shell_companies,
    detect_cycles,
    build_cluster_summary,
)

## Загрузка графа

In [ ]:
# Загружаем отфильтрованный граф из pickle (локальная ФС)
graph_path = config.OUTPUT_FILTERED_GRAPH_PICKLE
print(f'Loading graph from: {graph_path}')

with open(graph_path, 'rb') as f:
    G = pickle.load(f)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## Leiden кластеризация

Алгоритм **Leiden** (CPM — Constant Potts Model) находит сообщества в графе — группы узлов, которые связаны между собой сильнее, чем с остальным графом. Параметр **gamma** контролирует гранулярность: выше gamma → меньше кластеров, крупнее группы.

Автоматический подбор gamma: перебираем несколько значений и выбираем по лучшей модулярности.

**Интерпретация**: каждый кластер — это потенциальная бизнес-группа или цепочка связанных компаний.

In [ ]:
membership, best_gamma = run_leiden_clustering(G)

# Применяем к графу
for node, cluster_id in membership.items():
    if node in G:
        G.nodes[node]['cluster'] = cluster_id

n_clusters = len(set(v for v in membership.values() if v >= 0))
print(f'\nBest gamma: {best_gamma}')
print(f'Clusters found: {n_clusters}')

In [ ]:
# Распределение размеров кластеров
cluster_sizes = pd.Series(membership).value_counts().sort_index()
cluster_sizes = cluster_sizes[cluster_sizes.index >= 0]  # exclude unassigned
print(f'\nCluster size distribution:')
print(cluster_sizes.describe())

fig, ax = plt.subplots(figsize=(10, 4))
cluster_sizes.plot(kind='bar', ax=ax)
ax.set_xlabel('Cluster ID')
ax.set_ylabel('Members')
ax.set_title('Cluster Size Distribution')
plt.tight_layout()
plt.show()

## Метрики центральности

Центральность показывает, насколько узел «важен» в графе:

| Метрика | Что показывает | Высокое значение = |
|---------|---------------|--------------------|
| **PageRank** | Общая важность (учитывает входящие связи рекурсивно) | Ключевая компания, на которую «указывают» многие |
| **Betweenness** | Посредничество (как часто узел лежит на кратчайших путях) | Потенциальный «привратник» или транзитная компания |
| **Clustering coefficient** | Локальная связность (связаны ли соседи между собой) | Тесный клуб партнёров |
| **Flow-through ratio** | Доля транзитного потока (min(in, out) / max(in, out)) | Близко к 1.0 = деньги «проходят через» компанию |

In [ ]:
metrics_df = compute_centrality(G)

# Применяем к графу
for node_id, row in metrics_df.iterrows():
    if node_id in G:
        G.nodes[node_id]['pagerank'] = row['pagerank']
        G.nodes[node_id]['betweenness'] = row['betweenness']

print('Top 10 by PageRank:')
top_pr = metrics_df.nlargest(10, 'pagerank')[['pagerank', 'betweenness', 'in_degree', 'out_degree', 'total_in_flow']]
display(top_pr)

print('\nTop 10 by Betweenness:')
top_bc = metrics_df.nlargest(10, 'betweenness')[['pagerank', 'betweenness', 'clustering_coef', 'flow_through_ratio']]
display(top_bc)

## Классификация ролей

Каждому узлу присваивается одна из пяти ролей на основе метрик:

| Роль | Описание | Типичные признаки |
|------|----------|-------------------|
| **parent** | Головная компания | Высокий PageRank, много исходящих связей |
| **subsidiary** | Дочерняя компания | Высокая зависимость от одного контрагента (top_k_concentration → 1) |
| **conduit** | Транзитная компания | Высокий flow_through_ratio, высокий betweenness |
| **shell** | Shell-компания | Нет зарплатных выплат, высокий оборот через мало контрагентов |
| **regular** | Обычная компания | Не подходит под другие роли |

## Классификация ролей

In [ ]:
## Shell-компании

Автоматическое обнаружение потенциальных shell-компаний на основе комбинации сигналов:
- **Нет зарплатных выплат** (has_salary_payments = False) — нет реальных сотрудников
- **Высокий flow_through_ratio** — деньги проходят транзитом
- **Низкий clustering coefficient** — слабые связи с окружением
- **Низкий PageRank** при высоком обороте — «невидимая» компания с большим потоком

Shell-score > порога (по умолчанию 0.5) → компания считается подозрительной.

In [ ]:
metrics_df = classify_node_roles(metrics_df, G)

# Применяем роли к графу
for node_id, row in metrics_df.iterrows():
    if node_id in G:
        G.nodes[node_id]['role'] = row['role']

print('Role distribution:')
display(metrics_df['role'].value_counts())

## Циклические платежи

Обнаружение замкнутых цепочек транзакций: A → B → C → A. Такие циклы могут указывать на:
- **Круговое финансирование** — искусственное наращивание оборотов
- **Обналичивание** — прогон средств через цепочку shell-компаний
- **Легитимные схемы** — закупочные циклы (поставщик → производитель → дистрибутор → поставщик)

Каждый цикл характеризуется длиной (количество участников) и общей суммой.

In [ ]:
shell_df = detect_shell_companies(metrics_df, G)

if len(shell_df) > 0:
    # Добавляем shell_score в основной metrics_df
    metrics_df['shell_score'] = 0.0
    for idx in shell_df.index:
        metrics_df.loc[idx, 'shell_score'] = shell_df.loc[idx, 'shell_score']
    
    print(f'Flagged shell companies: {len(shell_df)}')
    display_cols = ['pagerank', 'betweenness', 'clustering_coef', 'flow_through_ratio',
                    'has_salary_payments', 'shell_score', 'role']
    display(shell_df[[c for c in display_cols if c in shell_df.columns]])
else:
    metrics_df['shell_score'] = 0.0
    print('No shell companies flagged.')

## Сводка по кластерам

Итоговая таблица по каждому кластеру включает:
- **member_count** — количество участников
- **company_count / individual_count** — разбивка по типу
- **total_internal_turnover** — общий оборот внутри кластера
- **external_counterparty_count** — количество уникальных контрагентов за пределами кластера (чем больше — тем более «открытая» группа)
- **lead_company** — компания с наибольшим PageRank (предполагаемый центр группы)
- **has_cycles** — найдены ли циклические платежи внутри кластера
- **shell_count** — количество потенциальных shell-компаний

In [ ]:
cycles = detect_cycles(G)

if cycles:
    print(f'Cycles detected: {len(cycles)}')
    for i, cyc in enumerate(cycles[:10]):
        node_names = [G.nodes[n].get('name', str(n))[:20] for n in cyc['nodes']]
        print(f"  Cycle {i+1}: length={cyc['length']}, amount={cyc['total_amount']:,.0f}")
        print(f"    Nodes: {' → '.join(node_names)}")
else:
    print('No cycles detected.')

## Сводка по кластерам

In [ ]:
cluster_summary = build_cluster_summary(G, metrics_df, cycles)

if len(cluster_summary) > 0:
    display(cluster_summary)
else:
    print('No clusters to summarize.')

---

## Глоссарий терминов

| Термин | Описание |
|--------|----------|
| **Leiden (CPM)** | Алгоритм кластеризации графа; находит сообщества с высокой внутренней плотностью |
| **Gamma** | Параметр разрешения CPM: выше = более крупные кластеры |
| **PageRank** | Метрика важности узла; учитывает входящие связи рекурсивно (как Google ранжирует веб-страницы) |
| **Betweenness centrality** | Доля кратчайших путей, проходящих через узел; высокая = «привратник» |
| **Clustering coefficient** | Насколько соседи узла связаны между собой (0–1); высокий = тесный круг |
| **Flow-through ratio** | min(входящий поток, исходящий поток) / max(входящий, исходящий); близко к 1 = транзитная компания |
| **Edge Score** | Комплексная оценка важности связи (0–1): сумма + двусторонняя доля + важность узлов + стабильность |
| **Shell-компания** | Юрлицо с признаками фиктивности: нет сотрудников, высокий транзитный поток, слабые связи |
| **Shell score** | Числовая оценка «фиктивности» (0–1); выше порога = потенциальная shell-компания |
| **Циклический платёж** | Замкнутая цепочка транзакций (A→B→C→A); может указывать на схему |
| **external_counterparty_count** | Кол-во контрагентов кластера за его пределами; показывает «открытость» группы |

---

**Следующий шаг**: Откройте `04_visualization.ipynb` для интерактивной визуализации.

In [ ]:
# Сохраняем обновлённый граф с кластерами и метриками
graph_path = config.OUTPUT_FILTERED_GRAPH_PICKLE
with open(graph_path, 'wb') as f:
    pickle.dump(G, f)
print(f'Analyzed graph saved: {graph_path}')

# Метрики — Pandas Parquet (локальная ФС)
# Добавляем cluster_id к метрикам
metrics_df['cluster_id'] = metrics_df.index.map(lambda n: G.nodes[n].get('cluster', -1) if n in G else -1)
metrics_df.to_parquet(config.OUTPUT_GRAPH_METRICS)
print(f'Metrics saved: {config.OUTPUT_GRAPH_METRICS}')

if len(cluster_summary) > 0:
    cluster_summary.to_parquet(config.OUTPUT_CLUSTERS, index=False)
    print(f'Clusters saved: {config.OUTPUT_CLUSTERS}')

---

**Следующий шаг**: Откройте `04_visualization.ipynb` для интерактивной визуализации.